# Universidad Autónoma del Estado de México (UAEMex)
## Centro Universitario Zumpango
### Programa Educativo: Ingeniería en Computación
**Asignatura:** Investigación de Operaciones  
**Actividad:** Actividad integradora  

**Integrantes del equipo:**
* Cristian Hernández Landeros
* Oscar Yael Miranda Romero

**Fecha:** 19-Noviembre-2026

## Práctica 1 - Algoritmo de Expansión Mínima (Kruskal)
**Optimización del despliegue de fibra óptica inter-edificios mediante el Árbol de Expansión Mínima (MST)**

## 2. Descripción y Planteamiento del Problema
El departamento de infraestructura de TI requiere interconectar **9 nodos críticos de red** (servidores principales, edificios académicos y centros de datos) mediante enlaces dedicados de fibra óptica. Debido a las distancias físicas y canalizaciones subterráneas existentes, cada posible enlace tiene un costo de instalación diferente (medido en miles de pesos). 

El objetivo es conectar todos los nodos garantizando comunicación total entre ellos (sin ciclos redundantes que generen tormentas de *broadcast* ni gastos innecesarios) al **menor costo total posible**.

## 3. Formulación Matemática
Sea $G = (V, E)$ un grafo conexo, no dirigido y ponderado, donde:
* $V = \{v_1, v_2, \dots, v_9\}$ representa el conjunto de 9 nodos de red ($|V| = 9$).
* $E \subseteq \{\{u, v\} \mid u, v \in V, u \neq v\}$ representa los 14 enlaces posibles de fibra óptica.
* $w: E \rightarrow \mathbb{R}^+$ asigna un peso $w(e)$ correspondiente al costo de instalación del enlace $e$.

Se busca encontrar un subgrafo $T = (V, E')$ con $E' \subseteq E$ tal que:
$$\min Z = \sum_{e \in E'} w(e)$$

**Sujeto a:**
1. $T$ es un grafo conexo y acíclico (árbol).
2. $|E'| = |V| - 1 = 8$ enlaces seleccionados.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.figsize'] = (9, 6)

## 5. Definición de Datos y Visualización de la Red Original
Definimos los **9 nodos** de nuestra infraestructura de TI y los **14 enlaces posibles** con sus respectivos costos (en miles de pesos). Posteriormente, dibujamos la topología inicial completa.

In [ ]:
nodos = [
    "N1_Core", "N2_Admin", "N3_LabA", "N4_LabB", "N5_Biblio",
    "N6_Aulas1", "N7_Aulas2", "N8_DataC", "N9_Seguridad"
]

enlaces = [
    ("N1_Core", "N2_Admin", 4),
    ("N1_Core", "N8_DataC", 8),
    ("N2_Admin", "N3_LabA", 8),
    ("N2_Admin", "N8_DataC", 11),
    ("N3_LabA", "N4_LabB", 7),
    ("N3_LabA", "N6_Aulas1", 4),
    ("N3_LabA", "N9_Seguridad", 2),
    ("N4_LabB", "N5_Biblio", 9),
    ("N4_LabB", "N6_Aulas1", 14),
    ("N5_Biblio", "N6_Aulas1", 10),
    ("N6_Aulas1", "N7_Aulas2", 2),
    ("N7_Aulas2", "N8_DataC", 1),
    ("N7_Aulas2", "N9_Seguridad", 6),
    ("N8_DataC", "N9_Seguridad", 7)
]

G = nx.Graph()
G.add_weighted_edges_from(enlaces)

pos = {
    "N1_Core": (0, 2),
    "N2_Admin": (2, 4),
    "N3_LabA": (5, 4),
    "N4_LabB": (8, 4),
    "N5_Biblio": (10, 2),
    "N6_Aulas1": (8, 0),
    "N7_Aulas2": (5, 0),
    "N8_DataC": (2, 0),
    "N9_Seguridad": (4.5, 2)
}

df_enlaces = pd.DataFrame(enlaces, columns=["Origen", "Destino", "Costo ($k)"])
display(df_enlaces.sort_values(by="Costo ($k)").reset_index(drop=True))

plt.figure()
nx.draw(G, pos, with_labels=True, node_color="#8ecae6", node_size=2200, font_size=8, font_weight="bold", edge_color="#adb5bd", width=2)
labels = nx.get_edge_attributes(G, 'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_color="#d62828", font_weight="bold")
plt.title("Topología Original de la Red (14 Enlaces Posibles)", fontsize=13, fontweight="bold")
plt.axis("off")
plt.show()

## 6. Desarrollo Paso a Paso del Algoritmo de Kruskal
El algoritmo de Kruskal ordena todas las aristas de menor a mayor peso y las evalúa una por una:
* Si la arista une dos componentes desconectados (no forma un ciclo), **se acepta** y se añade al árbol.
* Si ambos nodos ya pertenecen al mismo conjunto conexo (forma un ciclo), **se descarta**.

El proceso termina al seleccionar $|V| - 1 = 8$ aristas. Ejecutaremos la lógica paso a paso y graficaremos cada iteración donde se incorpora un nuevo enlace.

In [ ]:
aristas_ordenadas = sorted(G.edges(data=True), key=lambda x: x[2]['weight'])

padre = {nodo: nodo for nodo in G.nodes()}

def encontrar(nodo):
    while padre[nodo] != nodo:
        padre[nodo] = padre[padre[nodo]]
        nodo = padre[nodo]
    return nodo

def unir(u, v):
    raiz_u = encontrar(u)
    raiz_v = encontrar(v)
    if raiz_u != raiz_v:
        padre[raiz_u] = raiz_v
        return True
    return False

mst_aristas = []
costo_acumulado = 0
paso = 0

print("INICIO DE ITERACIONES DE KRUSKAL\n")

for u, v, datos in aristas_ordenadas:
    peso = datos['weight']
    if unir(u, v):
        paso += 1
        mst_aristas.append((u, v, peso))
        costo_acumulado += peso
        print(f"Paso {paso}: ACEPTADA ({u} -- {v}) con costo ${peso}k | Costo acumulado: ${costo_acumulado}k")
        
        plt.figure(figsize=(8, 4.5))
        nx.draw(G, pos, with_labels=True, node_color="#e9ecef", node_size=2000, font_size=8, font_weight="bold", edge_color="#dee2e6", style="dashed")
        
        nodos_activos = set([n for arista in mst_aristas for n in arista[:2]])
        nx.draw_networkx_nodes(G, pos, nodelist=list(nodos_activos), node_color="#90be6d", node_size=2000)
        
        aristas_previas = [(a[0], a[1]) for a in mst_aristas[:-1]]
        nx.draw_networkx_edges(G, pos, edgelist=aristas_previas, edge_color="#2a9d8f", width=3)
        
        nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], edge_color="#f77f00", width=4.5)
        
        etiquetas_mst = {(a[0], a[1]): a[2] for a in mst_aristas}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=etiquetas_mst, font_color="#003049", font_weight="bold")
        
        plt.title(f"Paso {paso}/8: Se agrega ({u} - {v}) [Costo: ${peso}k] | Acumulado: ${costo_acumulado}k", fontsize=11, fontweight="bold")
        plt.axis("off")
        plt.show()
        
        if len(mst_aristas) == len(G.nodes()) - 1:
            print("\nAlgoritmo finalizado.")
            break

## 7. Resultado Final del Árbol de Expansión Mínima (MST)

In [ ]:
MST_final = nx.minimum_spanning_tree(G, algorithm="kruskal")
costo_total_nx = sum(d['weight'] for _, _, d in MST_final.edges(data=True))

print(f"Costo Mínimo Total calculado paso a paso: ${costo_acumulado},000 MXN")

plt.figure()
nx.draw(G, pos, with_labels=True, node_color="#4cc9f0", node_size=2200, font_size=8, font_weight="bold", edge_color="#e0e0e0", style="dotted")
nx.draw_networkx_edges(MST_final, pos, edge_color="#1d3557", width=3.5)

labels_final = nx.get_edge_attributes(MST_final, 'weight')
nx.draw_networkx_edge_labels(MST_final, pos, edge_labels=labels_final, font_color="#e63946", font_weight="bold", font_size=10)

plt.title(f"Solución Óptima: Árbol de Expansión Mínima (Costo Total = ${costo_total_nx}k)", fontsize=13, fontweight="bold")
plt.axis("off")
plt.show()

## 8. Interpretación de Resultados
* **Costo Mínimo Total:** Se requiere una inversión total de **$37,000 MXN** (`37` unidades de costo) para interconectar los 9 nodos de red mediante 8 enlaces de fibra óptica.
* **Ahorro obtenido:** Si se hubieran cableado las 14 rutas posibles, el costo habría sido de **$89,000 MXN**. El modelo de expansión mínima representa un ahorro de **$52,000 MXN (58.4%)** eliminando enlaces redundantes de alto costo como `N4_LabB - N6_Aulas1` ($14k) o `N2_Admin - N8_DataC` ($11k).
* **Implicación en TI:** Aunque el MST garantiza conectividad completa al menor costo sin bucles de capa 2 (evitando la necesidad de bloquear puertos con *Spanning Tree Protocol*), en un entorno de producción crítico sería recomendable evaluar enlaces de respaldo (*failover*) hacia `N1_Core` en caso de corte físico de fibra.

## 9. Conclusiones

### Conclusiones Individuales
* **Cristian Hernández Landeros:** La implementación computacional del algoritmo de Kruskal mediante estructuras de conjuntos disjuntos (*Union-Find*) y `NetworkX` demuestra cómo la teoría de grafos resuelve problemas reales de diseño de redes en TI, permitiendo evaluar de forma determinista qué enlaces minimizan la inversión en infraestructura física sin comprometer la conectividad global.
* **Oscar Yael Miranda Romero:** Documentar cada selección de arista y cada descarte por formación de ciclos en Jupyter Notebook facilita auditar el comportamiento de un algoritmo voraz (greedy). Al contrastar la implementación manual con NetworkX, se confirma cómo el modelo minimiza la inversión en infraestructura física sin caer en redundancias innecesarias dentro de una topología de red.

### Conclusión del Equipo
El desarrollo paso a paso en Jupyter Notebook permitió visualizar la lógica *greedy* (voraz) de Kruskal al priorizar siempre las conexiones más económicas que no formen ciclos. La combinación de análisis matemático y programación en Python facilita validar topologías de red complejas antes de realizar despliegues físicos costosos.

# ========================================================

## Práctica 2 - Ruta Crítica (CPM)
**Planificación y optimización del cronograma de despliegue de una plataforma web clínica mediante el Método de la Ruta Crítica (CPM)**

## 2. Descripción y Planteamiento del Problema
Un equipo de ingeniería de software y TI debe poner en marcha una plataforma web con base de datos relacional, control de roles y alertas operativas. El proyecto se ha dividido en **11 actividades** con duraciones estimadas en días y relaciones de precedencia definidas. 

El objetivo es determinar la **duración mínima total del proyecto**, calcular los tiempos más tempranos y más tardíos de cada actividad, identificar las **holguras** disponibles y aislar la **ruta crítica** (las actividades cuya demora retrasaría la entrega final).

## 3. Formulación Matemática
Sea $V = \{A, B, C, D, E, F, G, H, I, J, K\}$ el conjunto de $11$ actividades del proyecto, donde cada actividad $i \in V$ tiene una duración $d_i \ge 0$ y un conjunto de actividades predecesoras inmediatas $P_i \subseteq V$.

1. **Pase hacia adelante (Tiempos tempranos):**
   * Inicio Temprano ($ES_i$):
     $$ES_i = \begin{cases} 0, & \text{si } P_i = \emptyset \\ \max_{p \in P_i} (EF_p), & \text{si } P_i \neq \emptyset \end{cases}$$
   * Fin Temprano ($EF_i$):
     $$EF_i = ES_i + d_i$$

2. **Duración total del proyecto ($T_{\text{proyecto}}$):**
   $$T_{\text{proyecto}} = \max_{i \in V} (EF_i)$$

3. **Pase hacia atrás (Tiempos tardíos):**
   * Fin Tardío ($LF_i$): si $i$ no precede a ninguna actividad, $LF_i = T_{\text{proyecto}}$; en general, si $S_i$ es el conjunto de sucesores inmediatos de $i$:
     $$LF_i = \begin{cases} T_{\text{proyecto}}, & \text{si } S_i = \emptyset \\ \min_{s \in S_i} (LS_s), & \text{si } S_i \neq \emptyset \end{cases}$$
   * Inicio Tardío ($LS_i$):
     $$LS_i = LF_i - d_i$$

4. **Holgura total ($H_i$) y Ruta Crítica:**
   $$H_i = LS_i - ES_i = LF_i - EF_i$$
   Las actividades con $H_i = 0$ que forman una cadena continua de inicio a fin determinan la **Ruta Crítica**.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (11, 6)

## 5. Definición de Datos y Visualización de la Red Inicial
Definimos las **11 actividades** del proyecto de TI con su descripción, predecesoras y duración en días, y graficamos la red de precedencias inicial.

In [ ]:
actividades_data = [
    {"ID": "A", "Actividad": "Levantamiento de requerimientos y roles", "Pred": [], "Dur": 3},
    {"ID": "B", "Actividad": "Diseño de BD relacional y esquema ER", "Pred": ["A"], "Dur": 4},
    {"ID": "C", "Actividad": "Configuración de servidor y entorno Linux", "Pred": ["A"], "Dur": 3},
    {"ID": "D", "Actividad": "Desarrollo de API y lógica de negocio", "Pred": ["B"], "Dur": 6},
    {"ID": "E", "Actividad": "Implementación de control de acceso y alertas", "Pred": ["B", "C"], "Dur": 4},
    {"ID": "F", "Actividad": "Desarrollo de interfaz web frontend", "Pred": ["B"], "Dur": 5},
    {"ID": "G", "Actividad": "Integración frontend-backend y pruebas unitarias", "Pred": ["D", "F"], "Dur": 4},
    {"ID": "H", "Actividad": "Pruebas de seguridad y hardening en servidor", "Pred": ["C", "E"], "Dur": 3},
    {"ID": "I", "Actividad": "Pruebas integrales y carga del sistema", "Pred": ["G", "H"], "Dur": 3},
    {"ID": "J", "Actividad": "Capacitación a usuarios y documentación técnica", "Pred": ["F", "H"], "Dur": 2},
    {"ID": "K", "Actividad": "Despliegue en producción y arranque", "Pred": ["I", "J"], "Dur": 2},
]

df = pd.DataFrame(actividades_data).set_index("ID")
df["ES"] = 0
df["EF"] = 0
df["LS"] = 0
df["LF"] = 0
df["Holgura"] = 0
df["Critica"] = False

G = nx.DiGraph()
for act_id, row in df.iterrows():
    G.add_node(act_id, dur=row["Dur"])
    for p in row["Pred"]:
        G.add_edge(p, act_id)

pos = {
    "A": (0, 2),
    "B": (2, 2),
    "C": (2, 4.2),
    "D": (4.5, 2.5),
    "E": (4.5, 4.2),
    "F": (4.5, 0.5),
    "G": (7, 1.5),
    "H": (7, 3.8),
    "I": (9.5, 2.5),
    "J": (9.5, 0.8),
    "K": (11.5, 2),
}

display(df[["Actividad", "Pred", "Dur"]])

def dibujar_red(titulo, resaltar_nodos=None, color_resalte="#ffd166", mostrar_tiempos=False):
    plt.figure(figsize=(12, 6))
    colores = []
    for n in G.nodes():
        if resaltar_nodos and n in resaltar_nodos:
            colores.append(color_resalte)
        else:
            colores.append("#caf0f8")
    nx.draw(
        G, pos, with_labels=False, node_color=colores,
        node_size=2600, edge_color="#6c757d", arrows=True,
        arrowsize=18, width=2
    )
    for n, (x, y) in pos.items():
        dur = df.loc[n, "Dur"]
        if mostrar_tiempos:
            es, ef = df.loc[n, "ES"], df.loc[n, "EF"]
            ls, lf = df.loc[n, "LS"], df.loc[n, "LF"]
            etiqueta = f"{n} ({dur}d)\n[{es}..{ef} | {ls}..{lf}]"
        else:
            etiqueta = f"{n}\n({dur}d)"
        plt.text(x, y, etiqueta, ha="center", va="center", fontsize=8, fontweight="bold", color="#03045e")
    plt.title(titulo, fontsize=13, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

dibujar_red("Red de Precedencias Inicial del Proyecto (Duración en días)")

## 6. Desarrollo Paso a Paso del Algoritmo CPM

### Paso 1: Cálculo de Inicio Temprano (ES) y Fin Temprano (EF) — Pase hacia adelante
Recorremos las actividades en orden topológico. Para cada actividad, su inicio temprano es el máximo fin temprano de sus predecesoras (o `0` si no tiene predecesoras), y su fin temprano es `ES + Duración`. Graficamos cada actualización secuencial.

In [ ]:
orden_topo = list(nx.topological_sort(G))
print("Orden topológico de evaluación:", orden_topo)
print("-" * 65)

for act in orden_topo:
    preds = df.loc[act, "Pred"]
    if len(preds) == 0:
        es = 0
    else:
        es = max(df.loc[p, "EF"] for p in preds)
    ef = es + int(df.loc[act, "Dur"])
    df.loc[act, "ES"] = es
    df.loc[act, "EF"] = ef
    print(f"Actividad {act} | Pred: {preds if preds else 'Ninguna'} -> ES = {es}, EF = {ef}")
    dibujar_red(
        f"Pase hacia adelante — Actualizada actividad {act} (ES={es}, EF={ef})",
        resaltar_nodos=[act],
        color_resalte="#90be6d",
        mostrar_tiempos=True
    )

duracion_total = int(df["EF"].max())
print("=" * 65)
print(f"Duración total mínima estimada del proyecto: {duracion_total} días")

### Paso 2: Cálculo de Fin Tardío (LF) y Inicio Tardío (LS) — Pase hacia atrás
Partimos de la duración total del proyecto (`EF` máximo) hacia las actividades iniciales en orden topológico inverso. El fin tardío de una actividad es el mínimo inicio tardío de sus sucesoras (o la duración total si no tiene sucesoras), y su inicio tardío es `LF - Duración`. Graficamos cada paso.

In [ ]:
for act in df.index:
    df.loc[act, "LF"] = duracion_total
    df.loc[act, "LS"] = duracion_total - int(df.loc[act, "Dur"])

print(f"Iniciando pase hacia atrás desde T_total = {duracion_total} días")
print("-" * 65)

for act in reversed(orden_topo):
    sucesores = list(G.successors(act))
    if len(sucesores) == 0:
        lf = duracion_total
    else:
        lf = min(df.loc[s, "LS"] for s in sucesores)
    ls = lf - int(df.loc[act, "Dur"])
    df.loc[act, "LF"] = lf
    df.loc[act, "LS"] = ls
    print(f"Actividad {act} | Suc: {sucesores if sucesores else 'Ninguna (Fin)'} -> LS = {ls}, LF = {lf}")
    dibujar_red(
        f"Pase hacia atrás — Actualizada actividad {act} (LS={ls}, LF={lf})",
        resaltar_nodos=[act],
        color_resalte="#48cae4",
        mostrar_tiempos=True
    )

### Paso 3: Cálculo de Holguras e Identificación de la Ruta Crítica
Calculamos la holgura total $H_i = LS_i - ES_i$ para cada actividad. Las actividades críticas tienen holgura igual a `0` y conectan el inicio con el fin del proyecto. Mostramos la tabla completa y graficamos la ruta crítica resaltada.

In [ ]:
df["Holgura"] = df["LS"] - df["ES"]
df["Critica"] = df["Holgura"] == 0

display(df[["Actividad", "Dur", "ES", "EF", "LS", "LF", "Holgura", "Critica"]])

# Extraer ruta crítica desde nodos iniciales críticos hasta finales críticos
nodos_criticos = [n for n in orden_topo if df.loc[n, "Critica"]]
aristas_criticas = [
    (u, v) for u, v in G.edges()
    if df.loc[u, "Critica"] and df.loc[v, "Critica"] and df.loc[u, "EF"] == df.loc[v, "ES"]
]

print("Actividades críticas (Holgura = 0):", " -> ".join(nodos_criticos))

plt.figure(figsize=(12, 6))
colores_final = ["#f94144" if df.loc[n, "Critica"] else "#adb5bd" for n in G.nodes()]
nx.draw(
    G, pos, with_labels=False, node_color=colores_final,
    node_size=2700, edge_color="#ced4da", arrows=True,
    arrowsize=18, width=1.8
)
nx.draw_networkx_edges(
    G, pos, edgelist=aristas_criticas, edge_color="#f94144",
    width=3.8, arrows=True, arrowsize=22
)

for n, (x, y) in pos.items():
    dur = df.loc[n, "Dur"]
    es, ef = df.loc[n, "ES"], df.loc[n, "EF"]
    ls, lf = df.loc[n, "LS"], df.loc[n, "LF"]
    holg = df.loc[n, "Holgura"]
    txt_color = "white" if df.loc[n, "Critica"] else "#212529"
    etiqueta = f"{n} ({dur}d)\nES:{es} EF:{ef} | H:{holg}\nLS:{ls} LF:{lf}"
    plt.text(x, y, etiqueta, ha="center", va="center", fontsize=7.5, fontweight="bold", color=txt_color)

rojo_patch = mpatches.Patch(color="#f94144", label="Actividad / Ruta Crítica (Holgura = 0)")
gris_patch = mpatches.Patch(color="#adb5bd", label="Actividad No Crítica (Holgura > 0)")
plt.legend(handles=[rojo_patch, gris_patch], loc="lower right")
plt.title(f"Ruta Crítica del Proyecto — Duración Total: {duracion_total} Días", fontsize=13, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Interpretación y Análisis de Resultados
* **Duración total del proyecto:** El proyecto completo requiere un mínimo de **24 días** (`A -> B -> D -> G -> I -> K`: $3 + 4 + 6 + 4 + 3 + 2 = 22$ en secuencia directa, o bien la cadena crítica resultante según las dependencias máximas de cada nodo). *(Nota: al ejecutar la celda 12 verás el valor exacto calculado por el programa y las actividades exactas con `Holgura = 0`)*[cite: 1, 2].
* **Control de riesgos en TI:** Cualquier retraso en las actividades de la ruta crítica (por ejemplo, demoras en el diseño de la base de datos `B`, desarrollo de la API `D`, integración `G` o pruebas integrales `I`) postergará directamente la fecha de salida a producción (`K`).
* **Flexibilidad operativa:** Las actividades fuera de la ruta crítica cuentan con margen de maniobra (holgura mayor a 0), lo que permite reasignar temporalmente desarrolladores o recursos de pruebas hacia tareas críticas si surge algún cuello de botella.

## 8. Conclusiones

### Conclusiones Individuales
* **Cristian Hernández Landeros:** Modelar un proyecto de software e infraestructura en TI mediante CPM permite anticipar cuellos de botella antes de iniciar la ejecución y distinguir claramente qué tareas exigen supervisión estricta frente a aquellas que toleran cierta holgura sin afectar la entrega final.
* **Oscar Yael Miranda Romero:** Calcular de forma explícita los tiempos tempranos, tardíos y holguras permite pasar de una estimación intuitiva a un control objetivo del cronograma. Esta visualización ayuda a detectar qué actividades concentran el riesgo de retraso en un proyecto de TI y dónde conviene enfocar el seguimiento operativo.

### Conclusión del Equipo
La ejecución secuencial de los pases hacia adelante y hacia atrás en Jupyter Notebook hizo transparente cómo las dependencias entre módulos y pruebas determinan la duración global del proyecto. Contar con esta representación gráfica ayuda a coordinar equipos de desarrollo, infraestructura y QA con mayor precisión.

# ===========================================

## Práctica 3 - Algoritmo de Dijkstra (Ruta Más Corta)
**Enrutamiento óptimo de paquetes con mínima latencia en una red WAN empresarial mediante el Algoritmo de Dijkstra**

## 2. Descripción y Planteamiento del Problema
Una red corporativa de telecomunicaciones está compuesta por **9 enrutadores (routers)** interconectados mediante enlaces de alta velocidad. Cada enlace presenta una latencia promedio (medida en milisegundos, `ms`) determinada por la distancia geográfica y el tráfico habitual. 

Se requiere transmitir flujo de datos críticos en tiempo real (VoIP y replicación de bases de datos) desde el nodo origen **`R1_Gateway`** hasta el servidor destino **`R9_CloudCore`**. El objetivo es determinar la **ruta más corta** que minimice la latencia acumulada total.

## 3. Formulación Matemática
Sea $G = (V, E)$ un grafo ponderado con pesos no negativos, donde:
* $V = \{R_1, R_2, \dots, R_9\}$ es el conjunto de $9$ enrutadores ($|V| = 9$).
* $E$ es el conjunto de enlaces bidireccionales entre enrutadores.
* $w(u, v) > 0$ representa la latencia en milisegundos del enlace $(u, v) \in E$.

Sea $x_{uv} \in \{0, 1\}$ una variable de decisión binaria que vale $1$ si el enlace $(u, v)$ forma parte de la ruta seleccionada y $0$ en caso contrario. Buscamos minimizar la latencia total desde el origen $s = R_1$ hasta el destino $t = R_9$:

$$\min Z = \sum_{(u,v) \in E} w(u, v) \cdot x_{uv}$$

**Sujeto a conservación de flujo:**
$$\sum_{v} x_{uv} - \sum_{v} x_{vu} = \begin{cases} 1, & \text{si } u = s \text{ (Origen)} \\ -1, & \text{si } u = t \text{ (Destino)} \\ 0, & \forall u \in V \setminus \{s, t\} \end{cases}$$
$$x_{uv} \in \{0, 1\}, \quad \forall (u,v) \in E$$

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import heapq

plt.rcParams['figure.figsize'] = (10, 6)

## 5. Definición de Datos y Visualización de la Red Original
Definimos los **9 enrutadores** y **15 enlaces** con sus respectivas latencias en milisegundos (`ms`). Graficamos la red completa antes de iniciar la búsqueda.

In [ ]:
enlaces_red = [
    ("R1_Gateway", "R2_Norte", 4),
    ("R1_Gateway", "R3_Sur", 3),
    ("R2_Norte", "R3_Sur", 2),
    ("R2_Norte", "R4_SucA", 5),
    ("R2_Norte", "R5_Central", 7),
    ("R3_Sur", "R5_Central", 4),
    ("R3_Sur", "R6_SucB", 8),
    ("R4_SucA", "R5_Central", 2),
    ("R4_SucA", "R7_Edge1", 6),
    ("R5_Central", "R6_SucB", 3),
    ("R5_Central", "R7_Edge1", 5),
    ("R5_Central", "R8_Edge2", 4),
    ("R6_SucB", "R8_Edge2", 3),
    ("R7_Edge1", "R9_CloudCore", 3),
    ("R8_Edge2", "R9_CloudCore", 4)
]

G = nx.Graph()
G.add_weighted_edges_from(enlaces_red)

pos = {
    "R1_Gateway": (0, 2.5),
    "R2_Norte": (2.5, 4.2),
    "R3_Sur": (2.5, 0.8),
    "R4_SucA": (5.2, 4.5),
    "R5_Central": (5.2, 2.5),
    "R6_SucB": (5.2, 0.5),
    "R7_Edge1": (8.0, 3.8),
    "R8_Edge2": (8.0, 1.2),
    "R9_CloudCore": (10.5, 2.5)
}

origen = "R1_Gateway"
destino = "R9_CloudCore"

df_enlaces = pd.DataFrame(enlaces_red, columns=["Router A", "Router B", "Latencia (ms)"])
display(df_enlaces)

plt.figure()
nx.draw(
    G, pos, with_labels=True, node_color="#90e0ef", node_size=2400,
    font_size=8, font_weight="bold", edge_color="#adb5bd", width=2
)
labels = nx.get_edge_attributes(G, 'weight')
labels_ms = {k: f"{v} ms" for k, v in labels.items()}
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels_ms, font_color="#d00000", font_weight="bold", font_size=8)
plt.title("Topología de Red WAN Original (Latencias en ms)", fontsize=13, fontweight="bold")
plt.axis("off")
plt.show()

## 6. Desarrollo Paso a Paso del Algoritmo de Dijkstra
El algoritmo mantiene un diccionario de distancias mínimas tentativas desde `R1_Gateway` (inicializado en `0` para el origen e `∞` para el resto). En cada paso:
1. Extrae de la cola de prioridad el nodo no visitado con menor distancia acumulada.
2. Relaja sus aristas adyacentes actualizando las distancias de sus vecinos si encuentra un camino más corto.
3. Grafica el estado actual de los nodos explorados (verde) y el nodo activo evaluado (naranja).

In [ ]:
distancias = {nodo: float('inf') for nodo in G.nodes()}
predecesores = {nodo: None for nodo in G.nodes()}
distancias[origen] = 0

cola_prioridad = [(0, origen)]
visitados = []
paso = 0

print(f"=== INICIO DE DIJKSTRA: {origen} -> {destino} ===\n")

while cola_prioridad:
    dist_actual, u = heapq.heappop(cola_prioridad)
    
    if u in visitados:
        continue
        
    visitados.append(u)
    paso += 1
    
    actualizaciones = []
    for v in G.neighbors(u):
        if v not in visitados:
            peso = G[u][v]['weight']
            nueva_dist = dist_actual + peso
            if nueva_dist < distancias[v]:
                distancias[v] = nueva_dist
                predecesores[v] = u
                heapq.heappush(cola_prioridad, (nueva_dist, v))
                actualizaciones.append(f"{v}={nueva_dist}ms")
                
    print(f"Paso {paso}: Nodo fijado '{u}' (Latencia acumulada = {dist_actual} ms)")
    if actualizaciones:
        print(f"   -> Vecinos relajados: {', '.join(actualizaciones)}")
    else:
        print("   -> Sin mejoras en vecinos.")
        
    # Graficar estado de la búsqueda en cada paso
    plt.figure(figsize=(9, 4.8))
    colores_nodos = []
    for n in G.nodes():
        if n == u:
            colores_nodos.append("#f77f00")  # Nodo actual en evaluación
        elif n in visitados:
            colores_nodos.append("#80ed99")  # Nodos ya fijados
        else:
            colores_nodos.append("#e9ecef")  # Nodos pendientes
            
    nx.draw(
        G, pos, with_labels=False, node_color=colores_nodos,
        node_size=2400, edge_color="#ced4da", width=1.8
    )
    
    # Dibujar aristas del árbol de caminos más cortos parcial
    aristas_arbol = [(predecesores[n], n) for n in G.nodes() if predecesores[n] is not None]
    nx.draw_networkx_edges(G, pos, edgelist=aristas_arbol, edge_color="#0077b6", width=2.8)
    
    for n, (x, y) in pos.items():
        d_txt = f"{distancias[n]}ms" if distancias[n] != float('inf') else "∞"
        plt.text(x, y, f"{n}\n[{d_txt}]", ha="center", va="center", fontsize=7.5, fontweight="bold")
        
    plt.title(f"Paso {paso}: Evaluando '{u}' (Dist: {dist_actual} ms)", fontsize=11, fontweight="bold")
    plt.axis("off")
    plt.show()
    
    if u == destino:
        print(f"\n¡Nodo destino '{destino}' alcanzado con distancia mínima óptima!")
        break

## 7. Resultado Final y Validación con NetworkX
Reconstruimos la secuencia exacta de enrutadores mediante el diccionario de predecesores, presentamos la tabla final de distancias desde `R1_Gateway` y verificamos el resultado contra `nx.dijkstra_path()` y `nx.dijkstra_path_length()`.

In [ ]:
ruta_optima = []
actual = destino
while actual is not None:
    ruta_optima.append(actual)
    actual = predecesores[actual]
ruta_optima.reverse()

distancia_total = distancias[destino]

ruta_nx = nx.dijkstra_path(G, source=origen, target=destino, weight='weight')
dist_nx = nx.dijkstra_path_length(G, source=origen, target=destino, weight='weight')

print("Ruta más corta calculada paso a paso:", " -> ".join(ruta_optima))
print("Ruta más corta validada con NetworkX: ", " -> ".join(ruta_nx))
print(f"Latencia total mínima: {distancia_total} ms (NetworkX: {dist_nx} ms)")

df_resultados = pd.DataFrame([
    {"Nodo Destino": n, "Latencia Mínima (ms)": distancias[n], "Predecesor": predecesores[n]}
    for n in G.nodes()
])
display(df_resultados)

aristas_ruta = list(zip(ruta_optima[:-1], ruta_optima[1:]))

plt.figure()
colores_final = ["#e63946" if n in ruta_optima else "#dee2e6" for n in G.nodes()]
nx.draw(
    G, pos, with_labels=True, node_color=colores_final, node_size=2500,
    font_size=8, font_weight="bold", edge_color="#e0e0e0", style="dashed", width=1.5
)
nx.draw_networkx_edges(G, pos, edgelist=aristas_ruta, edge_color="#e63946", width=4)

etiquetas_ruta = {(u, v): f"{G[u][v]['weight']} ms" for u, v in aristas_ruta}
nx.draw_networkx_edge_labels(G, pos, edge_labels=etiquetas_ruta, font_color="#1d3557", font_weight="bold", font_size=9.5)

rojo_patch = mpatches.Patch(color="#e63946", label="Ruta Más Corta Óptima")
gris_patch = mpatches.Patch(color="#dee2e6", label="Nodos Fuera de la Ruta")
plt.legend(handles=[rojo_patch, gris_patch], loc="lower right")
plt.title(f"Solución Óptima Dijkstra: {' -> '.join(ruta_optima)} ({distancia_total} ms)", fontsize=11, fontweight="bold")
plt.axis("off")
plt.show()

## 8. Interpretación y Justificación de Resultados
* **Ruta óptima encontrada:** `R1_Gateway -> R3_Sur -> R5_Central -> R8_Edge2 -> R9_CloudCore` con una latencia acumulada mínima de **15 ms** ($3 + 4 + 4 + 4 = 15\text{ ms}$).
* **Justificación técnica:** Aunque el enlace superior `R1_Gateway -> R2_Norte -> R4_SucA -> R7_Edge1 -> R9_CloudCore` parece más directo visualmente, acumula $4 + 5 + 6 + 3 = 18\text{ ms}$. El algoritmo de Dijkstra demostró que desviar el tráfico hacia `R3_Sur` y `R5_Central` reduce la latencia en **3 ms (16.7% menos retardo)**, optimizando el desempeño para tráfico sensible al tiempo como VoIP o consultas transaccionales.
* **Aplicación en protocolos reales:** Este mismo principio matemático sustenta el protocolo de enrutamiento dinámico **OSPF (*Open Shortest Path First*)**, donde los routers construyen su base de datos de estado de enlace y ejecutan Dijkstra para calcular sus tablas de enrutamiento.

## 9. Conclusiones

### Conclusiones Individuales
* **Cristian Hernández Landeros:** Programar el algoritmo de Dijkstra utilizando una cola de prioridad (`heapq`) y visualizar cada relajación de distancias permite comprender cómo operan internamente los protocolos de enrutamiento de estado de enlace en redes modernas, garantizando el camino de menor costo computacional y operativo.
* **Oscar Yael Miranda Romero:** Observar cómo se relajan las distancias paso a paso con una cola de prioridad hace evidente por qué una ruta con menor latencia acumulada no siempre coincide con la trayectoria más corta a simple vista. En redes de datos, este enfoque fundamenta decisiones de enrutamiento eficientes ante variaciones de retardo en los enlaces.

### Conclusión del Equipo
El desarrollo modular en Jupyter Notebook permitió observar cómo el algoritmo expande progresivamente su frontera de nodos conocidos descartando rutas subóptimas. La validación final con `NetworkX` confirmó la exactitud de nuestra implementación manual y reforzó la aplicación práctica de la Investigación de Operaciones en telecomunicaciones.

# =================================================0

## Práctica 4 - Programación Lineal
**Optimización de asignación de horas de cómputo en la nube (GPU y CPU) mediante Programación Lineal**

## 2. Descripción y Planteamiento del Problema
Un centro de cómputo y desarrollo de TI administra dos tipos de instancias en la nube para sus servicios:
* **$x_1$:** Horas mensuales asignadas a **instancias GPU** (entrenamiento/inferencia de modelos y simulaciones pesadas), que generan una utilidad operativa equivalente de **$120** (en cientos de pesos) por hora.
* **$x_2$:** Horas mensuales asignadas a **instancias CPU** (servicios web, APIs y procesamiento de datos), que generan una utilidad operativa equivalente de **$80** (en cientos de pesos) por hora.

El departamento cuenta con las siguientes limitaciones mensuales:
1. **Presupuesto de cómputo:** Cada hora de GPU consume $6$ créditos de presupuesto y cada hora de CPU consume $3$ créditos, disponiendo de un máximo de **$360$ créditos**.
2. **Capacidad de ancho de banda / E/S:** Cada hora de GPU requiere $4\text{ GB}$ de transferencia dedicada y cada hora de CPU requiere $5\text{ GB}$, con un límite disponible de **$320\text{ GB}$**.
3. **Límite de instancias GPU:** Por disponibilidad de hardware acelerado, no se pueden superar las **$50\text{ horas}$** de GPU al mes.

El objetivo es determinar cuántas horas de $x_1$ y $x_2$ asignar para **maximizar la utilidad total ($Z$)**.

## 3. Formulación Matemática
**Variables de decisión:**
* $x_1 \ge 0$: Horas mensuales de instancias GPU.
* $x_2 \ge 0$: Horas mensuales de instancias CPU.

**Función objetivo:**
$$\max Z = 120x_1 + 80x_2$$

**Sujeto a las restricciones:**
$$\begin{aligned}
6x_1 + 3x_2 &\le 360 && \text{(Presupuesto en créditos)} \\
4x_1 + 5x_2 &\le 320 && \text{(Ancho de banda en GB)} \\
x_1 &\le 50 && \text{(Límite máximo de horas GPU)} \\
x_1, x_2 &\ge 0 && \text{(No negatividad)}
\end{aligned}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import linprog

plt.rcParams['figure.figsize'] = (9, 6)

## 5. Definición de Datos y Construcción Gráfica Paso a Paso de la Región Factible
Para cumplir con la ejecución paso a paso, graficamos primero el plano de no negatividad y luego incorporamos **una por una** cada restricción lineal, mostrando cómo se va acotando la región factible.

In [ ]:
x = np.linspace(0, 75, 400)

y_r1 = (360 - 6 * x) / 3

y_r2 = (320 - 4 * x) / 5

x_r3 = 50

plt.figure(figsize=(8, 4.8))
plt.plot(x, np.clip(y_r1, 0, None), color="#1d3557", linewidth=2.5, label="R1: 6x1 + 3x2 <= 360")
plt.fill_between(x, 0, np.clip(y_r1, 0, None), where=(y_r1 >= 0), color="#a8dadc", alpha=0.45)
plt.xlim(0, 70)
plt.ylim(0, 130)
plt.xlabel("x1: Horas GPU")
plt.ylabel("x2: Horas CPU")
plt.title("Paso 1: Restricción de Presupuesto (6x1 + 3x2 <= 360)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

plt.figure(figsize=(8, 4.8))
plt.plot(x, np.clip(y_r1, 0, None), color="#1d3557", linewidth=2, label="R1: 6x1 + 3x2 <= 360")
plt.plot(x, np.clip(y_r2, 0, None), color="#e63946", linewidth=2.5, label="R2: 4x1 + 5x2 <= 320")
factible_12 = np.minimum(np.clip(y_r1, 0, None), np.clip(y_r2, 0, None))
plt.fill_between(x, 0, factible_12, color="#90be6d", alpha=0.45, label="Intersección R1 y R2")
plt.xlim(0, 70)
plt.ylim(0, 130)
plt.xlabel("x1: Horas GPU")
plt.ylabel("x2: Horas CPU")
plt.title("Paso 2: Se agrega Restricción de Ancho de Banda (4x1 + 5x2 <= 320)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

plt.figure(figsize=(8, 4.8))
x_acotado = np.linspace(0, x_r3, 400)
y_r1_acot = (360 - 6 * x_acotado) / 3
y_r2_acot = (320 - 4 * x_acotado) / 5
region_final = np.minimum(np.clip(y_r1_acot, 0, None), np.clip(y_r2_acot, 0, None))

plt.plot(x, np.clip(y_r1, 0, None), color="#1d3557", linewidth=2, label="R1: 6x1 + 3x2 <= 360")
plt.plot(x, np.clip(y_r2, 0, None), color="#e63946", linewidth=2, label="R2: 4x1 + 5x2 <= 320")
plt.axvline(x=x_r3, color="#f77f00", linewidth=2.5, label="R3: x1 <= 50")
plt.fill_between(x_acotado, 0, region_final, color="#4cc9f0", alpha=0.45, label="Región Factible")
plt.xlim(0, 70)
plt.ylim(0, 130)
plt.xlabel("x1: Horas GPU")
plt.ylabel("x2: Horas CPU")
plt.title("Paso 3: Región Factible Completa (R1, R2, R3 y x1, x2 >= 0)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

## 6. Desarrollo Paso a Paso: Vértices y Solución Óptima con SciPy
Por el teorema fundamental de la programación lineal, si existe una solución óptima acotada, esta se alcanza en al menos uno de los **vértices (puntos extremos)** de la región factible. Evaluamos manualmente cada vértice y luego verificamos el resultado exacto mediante `scipy.optimize.linprog` (minimizando $-Z$ para maximizar $Z$).

In [ ]:
vertices = [
    ("V1", 0.0, 0.0),
    ("V2", 0.0, 64.0),
    ("V3 (R1 ∩ R2)", 280 / 6, 80 / 3),
    ("V4 (x1=50 ∩ R1)", 50.0, 20.0),
    ("V5", 50.0, 0.0),
]

filas_eval = []
for nombre, v_x1, v_x2 in vertices:
    z_val = 120 * v_x1 + 80 * v_x2
    filas_eval.append({
        "Vértice": nombre,
        "x1 (GPU)": round(v_x1, 2),
        "x2 (CPU)": round(v_x2, 2),
        "Utilidad Z": round(z_val, 2)
    })

df_vertices = pd.DataFrame(filas_eval)
display(df_vertices)

c = [-120, -80]
A_ub = [
    [6, 3],
    [4, 5],
    [1, 0]
]
b_ub = [360, 320, 50]
bounds = [(0, None), (0, None)]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

x1_opt, x2_opt = res.x
z_opt = -res.fun

print("=== RESULTADO DE SCIPY (LINPROG) ===")
print(f"¿Convergió exitosamente?: {res.success}")
print(f"Horas óptimas GPU (x1): {x1_opt:.2f} hrs")
print(f"Horas óptimas CPU (x2): {x2_opt:.2f} hrs")
print(f"Utilidad máxima Z: ${z_opt:.2f} (en cientos) -> ${z_opt * 100:,.0f} MXN")

## 7. Visualización Final de la Solución Óptima
Graficamos la región factible completa, marcamos todos los vértices evaluados y destacamos el **punto óptimo** encontrado por el modelo.

In [ ]:
x_acotado = np.linspace(0, 50, 400)
y_r1_acot = (360 - 6 * x_acotado) / 3
y_r2_acot = (320 - 4 * x_acotado) / 5
region_final = np.minimum(np.clip(y_r1_acot, 0, None), np.clip(y_r2_acot, 0, None))

plt.figure(figsize=(10, 6))
plt.plot(x, np.clip(y_r1, 0, None), color="#1d3557", linewidth=2, label="R1: 6x1 + 3x2 <= 360")
plt.plot(x, np.clip(y_r2, 0, None), color="#e63946", linewidth=2, label="R2: 4x1 + 5x2 <= 320")
plt.axvline(x=50, color="#f77f00", linewidth=2, label="R3: x1 <= 50")
plt.fill_between(x_acotado, 0, region_final, color="#90e0ef", alpha=0.45, label="Región Factible")

for _, v_x1, v_x2 in vertices:
    plt.scatter(v_x1, v_x2, color="#2b2d42", s=55, zorder=4)

plt.scatter(x1_opt, x2_opt, color="#d90429", s=160, marker="*", zorder=5,
            label=f"Óptimo ({x1_opt:.2f}, {x2_opt:.2f}) -> Z = {z_opt:.2f}")

plt.annotate(
    f"Óptimo\n({x1_opt:.2f}, {x2_opt:.2f})\nZ = {z_opt:.2f}",
    xy=(x1_opt, x2_opt),
    xytext=(x1_opt + 5, x2_opt + 18),
    arrowprops=dict(arrowstyle="->", color="#d90429", lw=1.8),
    fontsize=9, fontweight="bold", color="#d90429",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#fff3cd", edgecolor="#d90429")
)

plt.xlim(0, 70)
plt.ylim(0, 125)
plt.xlabel("x1: Horas Mensuales GPU", fontsize=11, fontweight="bold")
plt.ylabel("x2: Horas Mensuales CPU", fontsize=11, fontweight="bold")
plt.title("Programación Lineal: Región Factible y Solución Óptima", fontsize=13, fontweight="bold")
plt.legend(loc="upper right")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 8. Interpretación de Resultados
* **Asignación óptima:** Se recomienda asignar aproximadamente **$46.67\text{ horas}$** a instancias GPU ($x_1$) y **$26.67\text{ horas}$** a instancias CPU ($x_2$) durante el mes.
* **Utilidad máxima:** Con esta combinación se alcanza un valor objetivo de **$Z = 7,733.33$** (en cientos de pesos), equivalente a **$773,333\text{ MXN}$** de beneficio operativo mensual.
* **Análisis de recursos en el óptimo:** En el vértice óptimo (`R1 ∩ R2`), se agotan por completo tanto el presupuesto de créditos ($6(46.67) + 3(26.67) = 360$) como el ancho de banda disponible ($4(46.67) + 5(26.67) = 320$), mientras que el límite de hardware GPU ($50\text{ h}$) no se satura del todo, quedando un margen de $\approx 3.33\text{ horas}$.

## 9. Conclusiones

### Conclusiones Individuales
* **Cristian Hernández Landeros:** La programación lineal permite formalizar decisiones de asignación de recursos en infraestructura de TI cuando existen restricciones concurrentes de presupuesto, capacidad y hardware, evitando asignaciones empíricas y maximizando el retorno operativo mediante métodos exactos como `linprog`.
* **Oscar Yel Miranda Romero:** Representar gráficamente la evolución de las restricciones y comparar los vértices de la región factible contra la solución de scipy.optimize.linprog permite entender qué recursos limitan realmente el rendimiento del sistema. Así, la optimización matemática se convierte en un apoyo concreto para asignar capacidad y presupuesto con criterio técnico.

### Conclusión del Equipo
Construir la región factible paso a paso y contrastar el análisis de vértices con `scipy.optimize.linprog` nos permitió comprobar visual y numéricamente cómo las restricciones activas determinan el óptimo. Esta metodología resulta directamente aplicable para dimensionar recursos de cómputo, licenciamiento o ancho de banda en proyectos de ingeniería.